# Analysis — Same Particle - Error Bar

Aqui analisamos mesma partícula, para 3 valores de potências (100mA, 150mA e 200mA) para 3 frequências diferenstes. Para cada (potência, freq) realizamos o experimento 20 vezes.

In [1]:
import matplotlib.pyplot as plt
import os
import numpy as np
np.seterr('warn')

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [2]:
from aux_functions import make_fitting_func, fit_curve, shorten_points, sort_experiments_by_freq
from treament_tcspc_data import treat_npy_array

## Configuração
Edite apenas esta célula para adicionar/remover partículas ou frequências.

In [3]:
data_path = "./sandbox/test_p10_laser700mA_counts4200"

In [4]:
filenames_all = []

for root, dirs, files in os.walk(data_path):
    # print(root)
    for name in files:
        # os.path.join combina o diretório atual (root) com o nome do arquivo
        # print(name)
        full_path = os.path.join(root, name)
        filenames_all.append(full_path.replace("\\", "/"))

In [5]:
def extract_info(path: str):
  freq = int(path.split("_fg")[1].split("Hz")[0])
  amp = float(path.split("Hz_")[1].split("V_")[0])
  offset = float(path.split("V_")[1].split("offs")[0])
  data = np.loadtxt(path) if ".txt" in path else np.load(path)

  return {
      "freq": freq,
      "amp": amp,
      "offset": offset,
      "data": data,
  }

In [6]:
data_txt = [extract_info(data_path_txt) for data_path_txt in filenames_all if ("step" in data_path_txt) and (".txt" in data_path_txt)]
data_npy = [extract_info(data_path_npy) for data_path_npy in filenames_all if ("step" in data_path_npy) and (".npy" in data_path_npy)]

In [7]:
data_test_npy = data_npy[0]["data"]
treated_data = treat_npy_array(data_test_npy)

In [8]:
# data_test_npy[0]

In [9]:
def get_data_test():
    xs = np.array([1, 2, 3])
    ys = 2*xs

    xs2 = np.array([1, 2, 3])
    ys2 = 3*xs2
    
    return np.array([[xs, ys], [xs2, ys2]])

data_testt = get_data_test()

In [ ]:
def get_mean_and_std(data):
    print(f"data: \n {data}")

    # Mean calculation
    xs = data[0][:,0]
    ys_sum = data[0][:,1]
    for idx, el in enumerate(data):
        if idx == 0:
            continue
        ys_sum += el[:,1]
    mean = ys_sum / len(data)
    print(f"mean for the y1: \n {mean[0]}")

    # Standard deviation calculation
    ys_sq_sum = data[0][:,1]**2
    for idx, el in enumerate(data):
        if idx == 0:
            continue
        ys_sq_sum += el[:,1]**2
    sq_mean = ys_sq_sum / len(data)
    # print(f"sq_mean: \n {sq_mean}")
    std = np.sqrt(sq_mean - mean**2)/np.sqrt(len(data))
    print(f"std for the y1: \n {std[0]}")

    # return np.column_stack((xs,mean))


# xs, ys = get_mean_and_std(data_test_npy).T
# plt.plot(xs, ys/max(ys), label="npy", color="orange")

# plt.plot(
#     data_txt[0]["data"][:,0], 
#     data_txt[0]["data"][:,1]/max(data_txt[0]["data"][:,1]), label="txt", color="blue")

# plt.show()
treated_data = treat_npy_array(data_testt)
get_mean_and_std(
    treated_data
)
# xs, ys = get_mean_and_std(data_testt).T
# plt.plot(xs, ys, label="test", color="orange")

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,6))

ax[0].plot(
    data_txt[0]["data"][:,0], 
    data_txt[0]["data"][:,1]/max(data_txt[0]["data"][:,1]), label="txt")
# a[0].legend()
for idx, el in enumerate(treated_data):
    ax[1].plot(
        treated_data[idx][:,0], 
        treated_data[idx][:,1], label=f"npy {idx}"
        )

plt.show()

In [ ]:
exp_list = []
for exp in range(14):
    exp_list.append(
        [extract_info(lum_data_path) for lum_data_path in filenames_all if (f"/exp{exp+1}/" in lum_data_path) and ("step" in lum_data_path)]
    )

In [ ]:
fallback_p0 = {
    (11, 100): [1587.92259477, 2275.5144266, -81.46603027],
    (13, 100): [1587.92259477, 2275.5144266, -81.46603027]
    }

## Funções auxiliares

## Análise — loop sobre N partículas

In [ ]:
# Pré-carrega os dados de excitação uma única vez (são os mesmos para todas as partículas)
# exc_data = {freq: np.loadtxt(f"{data_path_exc}/{freq}hzAPDA.txt") for freq in freq_list}
exc_data = sort_experiments_by_freq(exc_data)

cols = 4
rows = len(exc_data) // cols + (1 if len(exc_data) % cols > 0 else 0)

results = []   # armazena resultados de cada partícula
# error_bars = [] # armazena erros para cada partícula
phase_diffs = []

for exp_idx, exp_data in enumerate(exp_list):
    # freq, freq_path = freq_dic["freq"], freq_dic["path"]k
    exp_label = f"Experiment {exp_idx+1}"
    # correct = False # adj_phase_correction[f_idx] if p_idx < len(adj_phase_correction) else False
    

    print(f"\n{'='*50}")
    print(f"Processing Experiment {exp_idx+1}")
    print(f"{'='*50}")

    omegas      = []
    pcov_lum_list = []
    pcov_exc_list = []

    # ---- Subplots de verificação visual ----
    fig, axs = plt.subplots(rows, cols, figsize=(4*cols, 3*rows), constrained_layout=True)
    fig.suptitle(f"{exp_label} — frequency fits", fontsize=14)

    for index, freq_data in enumerate(exp_data):
        freq = freq_data["freq"]
        # print(f"\nProcessing frequency {freq} (type{type(freq)}) Hz for {exp_label}")
        # i, j = divmod(index, cols)
        # print(f"(i, j) = ({i}, {j}) for freq {freq} Hz")
        omega = 2 * np.pi * freq*1e-6  # Convertendo freq de Hz para rad/µs
        omegas.append(omega)

        fitting_func = make_fitting_func(omega)
        p0 = fallback_p0.get((exp_idx, freq), None)
        lum_curve = freq_data["data"]

        assert exc_data[index]["freq"] == freq, f"Frequency mismatch for index {index}!"
        exc_curve = exc_data[index]["data"]

        # print(f"Excitation curve for {freq} Hz: {exc_curve}")
        xs_exc, popt_exc = fit_curve(exc_curve, fitting_func, p0=p0)
        xs_lum, popt_lum = fit_curve(lum_curve, fitting_func, p0=p0)
        # pcov_exc_list.append(np.diag(pcov_exc)[2])
        # pcov_lum_list.append(np.diag(pcov_lum)[2])
        

        ph_exc = popt_exc[2]
        ph_lum = popt_lum[2]
        phase_diffs.append({
            "exp": exp_idx+1,
            "freq": freq,
            "pd": 360 - (np.degrees(ph_lum - ph_exc) % 360) #np.degrees(ph_lum - ph_exc) # ,
            })

        # Plot normalizado
        x_exc, y_exc = shorten_points(exc_curve)
        x_lum, y_lum = shorten_points(lum_curve)
   
        
        ax = axs[index]
        ax.plot(x_lum, y_lum / np.max(y_lum), label="lum")
        ax.plot(xs_lum, fitting_func(xs_lum, *popt_lum) / np.max(y_lum), "g--", label="fit lum")
        ax.set_title(f"{freq} Hz")
        ax.legend(loc="upper left", fontsize=7)

        ax2 = ax.twinx()
        ax2.plot(x_exc, y_exc / np.max(y_exc), color="tab:orange", label="exc")
        ax2.plot(xs_exc, fitting_func(xs_exc, *popt_exc) / np.max(y_exc), "r--", label="fit exc")
        ax2.legend(loc="upper right", fontsize=7)

    plt.show()


## Comparison

In [ ]:
freq_list = sorted(list(set(
    [pd["freq"] for pd in phase_diffs]
)))

error_bars= []

for freq in freq_list:
    pd_for_freq = [pd["pd"] for pd in phase_diffs if pd["freq"] == freq]
    mean_pd = np.mean(pd_for_freq)
    std_pd = np.std(pd_for_freq)/np.sqrt(len(exp_list))

    # print(f"Phase differences for {freq} Hz: {len(pd_for_freq)}")

    error_bars.append({
        "freq": freq,
        "mean": mean_pd,
        "std": std_pd,
        })

for idx, r in enumerate(error_bars):
    # color = colors[idx % len(colors)]
    plt.errorbar(
        freq_list[idx],
        error_bars[idx]["mean"],
        yerr=error_bars[idx]["std"],
        label=f"{error_bars[idx]["freq"]} Hz",
        fmt='o',
        capsize=5,
        # color=color,
    )

plt.xscale("log")
plt.title("Phase Difference Comparison")
plt.ylabel("Degrees (º)")
plt.xlabel("f (Hz)")
plt.legend()
plt.show()

In [ ]:
error_bars